Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_3/0001/0001_1_1_2_Augmented.png'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)


Preprocessing for Training

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]
PROTOCOL1_TRAIN_INDICES = {
    1: [1, 2, 3],
    2: [1, 2, 3],
    3: [1, 2, 3],
    4: [1, 2]
}

train_data = []
train_labels = []

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    print(f"🔧 Applying denoising (h={h})...")
    return cv2.fastNlMeansDenoising(image, h=h)

# === FUNCTION TO FUSE FINGERS FOR P1 TRAINING ===
def fuse_fingers_p1(subject_path, subject_id):
    fused_samples = []
    labels = []

    sample_counter = 0

    for img_num, aug_list in PROTOCOL1_TRAIN_INDICES.items():
        for aug_id in aug_list:
            fused_vector = []
            complete = True
            sample_counter += 1
            print(f"\n📦 Subject {subject_id} — Augmented Sample #{sample_counter:02d} (from image{img_num}_{aug_id})")

            for finger in FINGER_NUMS:
                fname = f"{subject_id}_{finger}_{img_num}_{aug_id}_Augmented.png"
                img_path = os.path.join(subject_path, fname)
                print(f"🖼️ Loading image: {img_path}")

                if not os.path.exists(img_path):
                    print(f"⚠️ Missing image: {img_path}")
                    complete = False
                    break

                # === Step 1: Load & Resize ===
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, IMAGE_SIZE)
                print(f"📏 Resized to: {IMAGE_SIZE}")

                # === Step 2: Denoising ===
                img_denoised = apply_denoising(img, h=10)
                print("🔧 Denoising applied.")

                # === Step 3: Histogram Equalization ===
                img_eq = exposure.equalize_hist(img_denoised)
                print("✨ Histogram equalization applied.")

                # === Step 4: Normalization ===
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
                print(f"📊 Normalized (mean ≈ 0, std ≈ 1): mean={np.mean(img_norm):.2f}, std={np.std(img_norm):.2f}")

                # === Step 5: Flatten & Collect ===
                fused_vector.append(img_norm.flatten())
                print(f"✅ Finger {finger} image processed.")


            if complete and len(fused_vector) == 6:
                fused = np.concatenate(fused_vector)
                label = f"{subject_id}_fused_aug{sample_counter:02d}"
                fused_samples.append(fused)
                labels.append(label)
                print(f"✅ Fused sample created and labeled: {label}")
            else:
                print(f"❌ Incomplete fusion for Subject {subject_id}, Sample #{sample_counter:02d}")

    return fused_samples, labels

# === LOAD TRAINING DATA FOR ALL SUBJECTS ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="🔄 Loading Protocol 1 - Strategy 1 Training"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    print(f"\n🔍 Processing Subject Folder: {subject_path}")
    fused_images, fused_labels = fuse_fingers_p1(subject_path, subj)
    train_data.extend(fused_images)
    train_labels.extend(fused_labels)

# === CONVERT TO NUMPY ARRAYS ===
train_data = np.array(train_data)
train_labels = np.array(train_labels)

# === SHAPE CHECK ===
print("\n📊 ✅ Final Train Data Shape:", train_data.shape)
print("📌 ✅ Train Labels Shape:", train_labels.shape)
print("🧾 ✅ First Few Labels:", train_labels[:5])


Test Preprocessing

In [ ]:
import numpy as np

print("🔧 Starting PCA using Gram matrix...")

# === Step 1: Center the training data ===
print("📍 Centering training data...")
mean_vector = np.mean(train_data, axis=0)
centered_data = train_data - mean_vector  # Shape: (n_samples, n_features)

# === Step 2: Compute Gram matrix (subject-to-subject) ===
print("📐 Computing Gram matrix (size: subjects × subjects)...")
gram_matrix = centered_data @ centered_data.T  # Shape: (n_samples, n_samples)

# === Step 3: Eigen decomposition of Gram matrix ===
print("🧮 Performing eigen-decomposition of Gram matrix...")
eig_vals, eig_vecs = np.linalg.eigh(gram_matrix)  # Use eigh for symmetric matrix

# === Step 4: Sort eigenvalues and eigenvectors in descending order ===
print("📊 Sorting eigenvalues and eigenvectors...")
sorted_indices = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[sorted_indices]
eig_vecs = eig_vecs[:, sorted_indices]

# === Step 5: Filter valid eigenvalues (> 1e-10 for stability) ===
print("✅ Filtering out near-zero eigenvalues...")
valid_mask = eig_vals > 1e-10
eig_vals_valid = eig_vals[valid_mask]
eig_vecs_valid = eig_vecs[:, valid_mask]
print(f"ℹ️ Retained {len(eig_vals_valid)} valid eigenvalues.")

# === Step 6: Project eigenvectors back to original feature space ===
print("↩️ Mapping eigenvectors to original feature space...")
eig_vecs_full = (centered_data.T @ eig_vecs_valid) / np.sqrt(eig_vals_valid)

# === Step 6.5: Normalize eigenvectors (optional but recommended) ===
print("🔄 Normalizing eigenvectors...")
eig_vecs_full = eig_vecs_full / np.linalg.norm(eig_vecs_full, axis=0)

# === Step 7: Project training data into PCA space ===
print("📤 Projecting centered data to PCA space...")
train_data_pca = centered_data @ eig_vecs_full  # Shape: (n_samples, n_components)

# === Final Outputs ===
print("\n✅ PCA projection completed successfully.")
print("📐 Transformed training data shape:", train_data_pca.shape)
print("📊 Number of principal components used:", eig_vecs_full.shape[1])


Test:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]

# === PROTOCOL P1 TEST SET ===
# Test = image4_3_Augmented + original images 1, 2, 3, 4
TEST_FILES = [
    ("4", "3_Augmented"),  # one augmented
    ("1", ""), ("2", ""), ("3", ""), ("4", "")  # 4 original images
]

test_data = []
test_labels = []

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    print(f"🔧 Applying denoising (h={h})...")
    return cv2.fastNlMeansDenoising(image, h=h)

# === TEST LOADING LOOP ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="🧪 Loading Protocol P1 - Strategy 1 Test"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for img_num, suffix in TEST_FILES:
        fused_vector = []
        complete = True
        img_label = f"{img_num}_{suffix}" if suffix else f"{img_num}_orig"
        print(f"\n📦 Subject {subj} — Test Image {img_label}")

        for finger in FINGER_NUMS:
            if suffix:
                fname = f"{subj}_{finger}_{img_num}_{suffix}.png"
            else:
                fname = f"{subj}_{finger}_{img_num}.png"

            img_path = os.path.join(subject_path, fname)
            print(f"🖼️ Loading: {img_path}")

            if not os.path.exists(img_path):
                print(f"⚠️ Missing image: {img_path}")
                complete = False
                break

            # === Step 1: Load & Resize ===
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, IMAGE_SIZE)
            print(f"📏 Resized to: {IMAGE_SIZE}")

            # === Step 2: Denoising ===
            img_denoised = apply_denoising(img, h=10)
            print("🔧 Denoising applied.")

            # === Step 3: Histogram Equalization ===
            img_eq = exposure.equalize_hist(img_denoised)
            print("✨ Histogram equalization applied.")

            # === Step 4: Normalization ===
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
            print(f"📊 Normalized (mean ≈ 0, std ≈ 1): mean={np.mean(img_norm):.2f}, std={np.std(img_norm):.2f}")

            # === Step 5: Flatten & Collect ===
            fused_vector.append(img_norm.flatten())
            print(f"✅ Finger {finger} image processed.")


        if complete and len(fused_vector) == 6:
            sample_vector = np.concatenate(fused_vector)
            test_data.append(sample_vector)
            label = f"{subj}_fused_test_{img_label}"
            test_labels.append(label)
            print(f"🎯 Test sample created and labeled: {label}")
        else:
            print(f"❌ Incomplete fusion for Subject {subj}, Test {img_label}")

# === CONVERT TO NUMPY ARRAYS ===
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# === SHAPE CHECK ===
print("\n✅ Test data loaded successfully.")
print(f"🧪 Total test samples: {len(test_data)}")
if len(test_data) > 0:
    print(f"📐 Example test vector shape: {test_data[0].shape}")
    print(f"🧾 Example test label: {test_labels[0]}")


Benchmarking

In [ ]:
correct_matches = 0
total_tests = len(test_data)

# === Step 1: Project test data to PCA space ===
centered_test_data = test_data - mean_vector
proj_test_data = centered_test_data @ eig_vecs_full
print("\n📤 Test data projected into PCA space.")

# === Step 2: Match each test sample against training set ===
for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_label = test_labels[i]  # e.g., "0001_fused_test_4_orig"

    # 📏 Manhattan distance between test and all training samples
    distances = np.sum(np.abs(train_data_pca - proj_test), axis=1)

    # 🏆 Nearest neighbor prediction
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., "0001_fused_aug01"

    # 🎯 Extract subject IDs for identity comparison
    true_id = true_label.split('_')[0]
    pred_id = predicted_label.split('_')[0]

    # ✅ Compare IDs
    if pred_id == true_id:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"🔍 Test {i+1:02d}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# === Final Accuracy ===
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Final Person Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")
